# **Inferencing with Detectron2 Pretrained Models**

<img src="https://dl.fbaipublicfiles.com/detectron2/Detectron2-Logo-Horz.png" width="500">

- **Installing Detectron2**

In [ ]:
!!!!python -m pip install pyyaml==5.1
import sys,os,distutils.core

# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities (e.g. compiled operators).
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions

!git clone 'https://github.com/facebookresearch/detectron2'

dist = distutils.core.run_setup("./detectron2/setup.py")

# Construct the installation arguments as a Python string first
_install_reqs = ' '.join(f"'{x}'" for x in dist.install_requires)
# Then pass the Python string to the shell command
!python -m pip install {_install_reqs}
sys.path.insert(0,os.path.abspath('./detectron2'))

Cloning into 'detectron2'...
remote: Enumerating objects: 16057, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 16057 (delta 25), reused 8 (delta 8), pack-reused 16016 (from 3)
Receiving objects: 100% (16057/16057), 6.83 MiB | 20.63 MiB/s, done.
Resolving deltas: 100% (11377/11377), done.
Ignoring dataclasses: markers 'python_version < "3.7"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.1/163.1 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.4/268.4 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch,detectron2
!nvcc --version

TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]

print("\nTORCH",TORCH_VERSION, "\nCUDA: ",CUDA_VERSION)
print("\nDETECTRON2:",detectron2.__version__)

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0

TORCH 2.11 
CUDA:  cu128

DETECTRON2: 0.6


- **IMPORTS**

In [ ]:
# SOME BASIC SETUP
# SETUP dtectron2 logger

import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

#import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog,DatasetCatalog

/content/detectron2/detectron2/model_zoo/model_zoo.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


- **Run Multiple pre-trained detectron2 models**

- **Explore Multiple models [LINK](https://github.com/facebookresearch/detectron2/blob/main/MODEL_ZOO.md#coco-object-detection-baselines)**

# **Inferencing with Faster RCNN Resnet50 Model**

- **Get Our Inference Image**

In [ ]:
import requests
import os

def download_image(url,filename):
  """Downloads an image from a given URL and saves it to a file using requests.

  Args:
    url: The URL of the image.
    filename: The name of the file to save the image to.
  """
  try:
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36'
    }

    response = requests.get(url,headers=headers,stream=True)
    response.raise_for_status()

    with open(filename,'wb') as f:
      for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

    print(f"Image sucessfully downloaded at {filename}")

  except requests.exceptions.RequestException as e:
    print(f"Error during downloading image: {e}")


download_image("https://images.pexels.com/photos/13872248/pexels-photo-13872248.jpeg", "pexels-photo-13872248.jpg")

Image sucessfully downloaded at pexels-photo-13872248.jpg


In [ ]:
#Read and Display the image
import cv2
from google.colab.patches import cv2_imshow

im = cv2.imread("/content/pexels-photo-13872248.jpg")

if im is not None:
  cv2_imshow(im)

else:
  print("Error: Could not read the image file.")

- ***Loading the Pretrained Model & Config File***

In [ ]:
cfg = get_cfg()

# add project-specific config (e.g., TensorMask) here if you're not running a model in detectron2's core library
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_1x.yaml"))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5 # set threshold for this model

# Find a model from detectron2's model zoo. We can use the https://dl.fbaipublicfiles... url as well
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_50_FPN_1x.yaml")
cfg.MODEL.DEVICE = "cpu"

predictor = DefaultPredictor(cfg)
outputs = predictor(im)

[09/10 02:40:49 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from https://dl.fbaipublicfiles.com/detectron2/COCO-Detection/faster_rcnn_R_50_FPN_1x/137257794/model_final_b275ba.pkl ...


model_final_b275ba.pkl: 167MB [00:00, 242MB/s]                           
/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0910 02:40:56.286000 926 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


In [ ]:
outputs

{'instances': Instances(num_instances=6, image_height=4160, image_width=6240, fields=[pred_boxes: Boxes(tensor([[4687.3472, 1212.6678, 6113.4902, 4159.4204],
         [2052.9211, 1504.3485, 3375.1562, 4119.3047],
         [ 135.6413, 1598.2573, 1566.0182, 4107.3555],
         [2971.8997, 1421.8940, 3905.8193, 4098.9663],
         [3795.1172, 1578.6820, 4749.1387, 4127.5542],
         [1403.1208, 1816.4042, 2151.4761, 4147.2188]])), scores: tensor([0.9989, 0.9989, 0.9984, 0.9968, 0.9939, 0.9933]), pred_classes: tensor([0, 0, 0, 0, 0, 0])])}

- **Let's look in the model output**

In [ ]:
# look at the outputs. See https://detectron2.readthedocs.io/tutorials/models.html#model-output-format for specification
print(outputs["instances"].pred_classes)
print(outputs["instances"].pred_boxes)
print(outputs["instances"].scores)

tensor([0, 0, 0, 0, 0, 0])
Boxes(tensor([[4687.3472, 1212.6678, 6113.4902, 4159.4204],
        [2052.9211, 1504.3485, 3375.1562, 4119.3047],
        [ 135.6413, 1598.2573, 1566.0182, 4107.3555],
        [2971.8997, 1421.8940, 3905.8193, 4098.9663],
        [3795.1172, 1578.6820, 4749.1387, 4127.5542],
        [1403.1208, 1816.4042, 2151.4761, 4147.2188]]))
tensor([0.9989, 0.9989, 0.9984, 0.9968, 0.9939, 0.9933])


In [ ]:
metadata = MetadataCatalog.get(cfg.DATASETS.TEST[0])
print(metadata.thing_classes)

classes = outputs['instances'].pred_classes
for id in classes:
  print(metadata.thing_classes[id])

['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
person
person
person
person
person
person


- **Visualizing the predicted image**

In [ ]:
# We can use `Visualizer` to draw the predictions on the image.
v = Visualizer(im[:,:,::-1],MetadataCatalog.get(cfg.DATASETS.TEST[0]),scale=1.2)
out = v.draw_instance_predictions(outputs['instances'].to('cpu'))
cv2_imshow(out.get_image()[:,:,::-1])

# **Inferencing with Faster RCNN Resnet101 Model**

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

im = cv2.imread("/content/drive/MyDrive/DL/Object Detection/data_annotation.png")
cv2_imshow(im)

Output hidden; open in https://colab.research.google.com to view.

- **Loading the Pretrained Model & Config File**

In [ ]:
cfg = get_cfg()

# add project-specific config
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml"))

# Set the thresh score
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

# Find a model from detectron2's model zoo. We can use the https://dl.fbaipublicfiles... url as well
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml")
cfg.MODEL.DEVICE = "cpu"

predictor = DefaultPredictor(cfg)
outputs = predictor(im)

[09/10 02:53:33 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from https://dl.fbaipublicfiles.com/detectron2/COCO-Detection/faster_rcnn_R_101_FPN_3x/137851257/model_final_f6e8b1.pkl ...


model_final_f6e8b1.pkl: 243MB [00:01, 234MB/s]                           


In [ ]:
outputs

{'instances': Instances(num_instances=68, image_height=1122, image_width=1402, fields=[pred_boxes: Boxes(tensor([[ 711.3051,  226.2698,  872.0497,  366.1501],
         [ 747.0477,  139.8427,  853.2754,  328.6999],
         [1261.4686, 1013.6521, 1346.2318, 1118.9595],
         [   4.5935,  161.1669,  391.9410,  342.8119],
         [ 337.5959,   19.3916,  378.1157,   62.8123],
         [ 833.1753,   53.5354, 1079.9656,  226.5536],
         [ 542.0110,  165.2850,  652.3540,  234.3802],
         [ 952.1171,  811.3948, 1202.3829, 1060.9967],
         [ 741.6403,  432.3296,  960.9882,  622.2247],
         [ 833.4368,  156.4532, 1016.1183,  275.4596],
         [1117.6111,  216.6585, 1205.8158,  365.3242],
         [ 434.4194,  897.4010,  574.3510, 1017.9569],
         [ 536.8890,  372.7600,  619.6140,  448.4855],
         [ 131.6548,  528.8456,  269.0143,  609.2841],
         [ 350.6886,  138.8006,  438.9413,  278.2885],
         [ 485.7644,  205.1640,  624.9115,  369.9084],
         [1032.8

- **Let's look in the model output**

In [ ]:
# look at the outputs. See https://detectron2.readthedocs.io/tutorials/models.html#model-output-format for specification
print(outputs["instances"].pred_classes)
print(outputs["instances"].pred_boxes)
print(outputs["instances"].scores)

tensor([ 1,  0,  0, 57, 74,  5, 63,  5, 19,  2,  3,  2, 58, 68, 58, 56,  2, 44,
        24,  2, 44,  0,  0,  2, 58, 18, 58, 49, 14, 72, 49,  3,  7,  2, 16,  2,
         0, 69,  2,  2,  2,  2,  0,  2,  7,  2, 44, 19, 17,  2,  2,  2,  2, 75,
         0, 73, 26, 44,  0,  0,  2, 75,  2, 73,  2,  7,  2, 73])
Boxes(tensor([[ 711.3051,  226.2698,  872.0497,  366.1501],
        [ 747.0477,  139.8427,  853.2754,  328.6999],
        [1261.4686, 1013.6521, 1346.2318, 1118.9595],
        [   4.5935,  161.1669,  391.9410,  342.8119],
        [ 337.5959,   19.3916,  378.1157,   62.8123],
        [ 833.1753,   53.5354, 1079.9656,  226.5536],
        [ 542.0110,  165.2850,  652.3540,  234.3802],
        [ 952.1171,  811.3948, 1202.3829, 1060.9967],
        [ 741.6403,  432.3296,  960.9882,  622.2247],
        [ 833.4368,  156.4532, 1016.1183,  275.4596],
        [1117.6111,  216.6585, 1205.8158,  365.3242],
        [ 434.4194,  897.4010,  574.3510, 1017.9569],
        [ 536.8890,  372.7600,  619.6140,

In [ ]:
metadat = MetadataCatalog.get(cfg.DATASETS.TEST[0])

classes = outputs['instances'].pred_classes

for id in classes:
  print(metadata.thing_classes[id])

bicycle
person
person
couch
clock
bus
laptop
bus
cow
car
motorcycle
car
potted plant
microwave
potted plant
chair
car
spoon
backpack
car
spoon
person
person
car
potted plant
sheep
potted plant
orange
bird
refrigerator
orange
motorcycle
truck
car
dog
car
person
oven
car
car
car
car
person
car
truck
car
spoon
cow
horse
car
car
car
car
vase
person
book
handbag
spoon
person
person
car
vase
car
book
car
truck
car
book


In [ ]:
v = Visualizer(im[:,:,::-1],MetadataCatalog.get(cfg.DATASETS.TEST[0]),scale=1.2)
out = v.draw_instance_predictions(outputs['instances'].to('cpu'))
cv2_imshow(out.get_image()[:,:,::-1])

Output hidden; open in https://colab.research.google.com to view.

# **Inferencing With RetinaNet R-50**

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

im = cv2.imread("/content/drive/MyDrive/DL/Object Detection/data_annotate.png")
cv2_imshow(im)

Output hidden; open in https://colab.research.google.com to view.

- **Load pretrained Model and Config file**

In [ ]:
cfg = get_cfg()

# add the project
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/retinanet_R_50_FPN_3x.yaml"))

# set the thresh score
cfg.MODEL.RETINANET.SCORE_THRESH_TEST = 0.5

# Find a model from detectron2's model zoo. We can use the https://dl.fbaipublicfiles... url as well
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/retinanet_R_50_FPN_3x.yaml")
cfg.MODEL.DEVICE = 'cpu'

predictor = DefaultPredictor(cfg)
outputs = predictor(im)

[09/10 03:19:26 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from https://dl.fbaipublicfiles.com/detectron2/COCO-Detection/retinanet_R_50_FPN_3x/190397829/model_final_5bd44e.pkl ...


model_final_5bd44e.pkl: 152MB [00:02, 71.7MB/s]                           
  pixel_mean
  pixel_std


In [ ]:
outputs

{'instances': Instances(num_instances=18, image_height=1024, image_width=1536, fields=[pred_boxes: Boxes(tensor([[ 437.6084,  193.7056,  885.3694,  561.3616],
         [ 392.7411,  353.8734,  593.6666,  697.3723],
         [  32.2312,  620.0363,  175.0912,  830.6010],
         [ 100.7975,  298.5857,  221.8314,  632.9839],
         [1331.4054,  328.8333, 1532.7477,  774.1885],
         [1434.1044,  404.8814, 1536.0000,  576.7766],
         [ 128.3893,  351.7606,  198.3843,  467.2655],
         [ 780.7257,  383.6385, 1200.2483,  683.3315],
         [ 394.0501,  473.9593,  583.7515,  793.8775],
         [1335.4850,  539.9931, 1536.0000,  890.6854],
         [ 561.3102,  305.7129,  645.3315,  390.2574],
         [ 897.0205,  419.3824,  972.6066,  485.0378],
         [ 291.4018,  407.1296,  344.9489,  495.2016],
         [ 289.7537,  367.3369,  344.9282,  477.5008],
         [ 337.0117,  381.0146,  434.9283,  468.6028],
         [ 242.0004,  373.8407,  298.5753,  427.9151],
         [1237.7

- **Let's look in the model output**

In [ ]:
# look at the outputs. See https://detectron2.readthedocs.io/tutorials/models.html#model-output-format for specification
print(outputs["instances"].pred_classes)
print(outputs["instances"].pred_boxes)
print(outputs["instances"].scores)

tensor([ 5,  0, 16,  0,  0, 24, 24,  2,  3,  1,  0,  0,  3,  0,  2,  2,  0,  2])
Boxes(tensor([[ 437.6084,  193.7056,  885.3694,  561.3616],
        [ 392.7411,  353.8734,  593.6666,  697.3723],
        [  32.2312,  620.0363,  175.0912,  830.6010],
        [ 100.7975,  298.5857,  221.8314,  632.9839],
        [1331.4054,  328.8333, 1532.7477,  774.1885],
        [1434.1044,  404.8814, 1536.0000,  576.7766],
        [ 128.3893,  351.7606,  198.3843,  467.2655],
        [ 780.7257,  383.6385, 1200.2483,  683.3315],
        [ 394.0501,  473.9593,  583.7515,  793.8775],
        [1335.4850,  539.9931, 1536.0000,  890.6854],
        [ 561.3102,  305.7129,  645.3315,  390.2574],
        [ 897.0205,  419.3824,  972.6066,  485.0378],
        [ 291.4018,  407.1296,  344.9489,  495.2016],
        [ 289.7537,  367.3369,  344.9282,  477.5008],
        [ 337.0117,  381.0146,  434.9283,  468.6028],
        [ 242.0004,  373.8407,  298.5753,  427.9151],
        [1237.7672,  383.4700, 1312.6110,  434.73

In [ ]:
metadata = MetadataCatalog.get(cfg.DATASETS.TEST[0])

classes = outputs['instances'].pred_classes

for id in classes:
  print(metadata.thing_classes[id])

bus
person
dog
person
person
backpack
backpack
car
motorcycle
bicycle
person
person
motorcycle
person
car
car
person
car


In [ ]:
v = Visualizer(im[:,:,::-1],MetadataCatalog.get(cfg.DATASETS.TEST[0]))
out = v.draw_instance_predictions(outputs['instances'].to('cpu'))
cv2_imshow(out.get_image()[:,:,::-1])

Output hidden; open in https://colab.research.google.com to view.